# Customer baseline — single-pass watermark verification

Rombak dari baseline customer `ucf101_robust_watermark_fixed.ipynb`.

Flow utama: **1 video → insert watermark sabila → Neural Codec → read watermark → ECC decode → SUCCESS/FAILED**.

H.264/H.265 dan retry tidak dijalankan. Retry hanya disediakan sebagai extension point.

In [ ]:
from pathlib import Path
import subprocess, tempfile
import numpy as np
import cv2
import torch
import torch.nn as nn

SEED = 42
FRAME_SIZE = (128, 128)
WATERMARK_TEXT = 'sabila'
ECC_REPEAT = 3
PAYLOAD_BITS = 8 * len(WATERMARK_TEXT)
WATERMARK_LENGTH = PAYLOAD_BITS * ECC_REPEAT
MAX_FRAMES = 64
CODEC_QUANT_LEVELS = 16

PROJECT_DIR = Path('/content/drive/MyDrive/Order/order_20260827_213103')
INPUT_VIDEO = PROJECT_DIR / 'input' / 'input.avi'
OUTPUT_DIR = PROJECT_DIR / 'output' / 'single_pass'
MODEL_DIR = PROJECT_DIR / 'output'
ENCODER_PATH = MODEL_DIR / 'encoder_finetune_v3.pth'
DECODER_PATH = MODEL_DIR / 'decoder_finetune_v3.pth'
NEURAL_CODEC_PATH = MODEL_DIR / 'neural_codec.pth'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED); np.random.seed(SEED)
print('Device:', device)
print('Input:', INPUT_VIDEO)

## 1. ECC customer baseline

Payload `sabila` = 48 bit. Customer memakai repetition ECC 3x → 144 encoded bits. Tidak ada pemotongan manual.

In [ ]:
def text_to_bits(text):
    return np.asarray([int(b) for ch in text for b in format(ord(ch), '08b')], dtype=np.int64)

def encode_ecc(text, repeat=ECC_REPEAT):
    payload = text_to_bits(text)
    return np.tile(payload, repeat), payload

def ecc_majority_vote(bits, payload_bits=PAYLOAD_BITS, repeat=ECC_REPEAT):
    arr = np.asarray(bits).reshape(-1)
    expected = payload_bits * repeat
    if arr.size < expected:
        raise ValueError(f'ECC bits kurang: butuh {expected}, dapat {arr.size}')
    copies = (arr[:expected] >= 0.5).astype(np.int64).reshape(repeat, payload_bits)
    return (copies.sum(axis=0) > repeat / 2).astype(np.int64)

def bits_to_text(bits):
    bits = np.asarray(bits).astype(np.int64).reshape(-1)
    return ''.join(
        chr(int(''.join(map(str, bits[i:i+8])), 2))
        if 32 <= int(''.join(map(str, bits[i:i+8])), 2) <= 126 else '�'
        for i in range(0, len(bits) - len(bits) % 8, 8)
    )

GROUND_TRUTH_BITS, PAYLOAD = encode_ecc(WATERMARK_TEXT)
GROUND_TRUTH_WM = torch.tensor([GROUND_TRUTH_BITS], dtype=torch.float32, device=device)
assert len(PAYLOAD) == 48 and len(GROUND_TRUTH_BITS) == 144
assert bits_to_text(ecc_majority_vote(GROUND_TRUTH_BITS)) == WATERMARK_TEXT

print('Payload bits:', len(PAYLOAD))
print('Encoded ECC :', len(GROUND_TRUTH_BITS))
print('Binary      :', ''.join(map(str, PAYLOAD)))
print('ECC decode  :', bits_to_text(ecc_majority_vote(GROUND_TRUTH_BITS)))

## 2. Customer watermark encoder / decoder

Mempertahankan arsitektur customer V2: channels=80, residual block tambahan, residual strength 0.18.

In [ ]:
class WatermarkEncoder(nn.Module):
    def __init__(self, wm_length=WATERMARK_LENGTH, channels=80):
        super().__init__()
        self.fc_wm = nn.Linear(wm_length, 16 * 8 * 8)
        self.wm_upsample = nn.Sequential(
            nn.ConvTranspose2d(16,32,4,2,1), nn.GroupNorm(8,32), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32,16,4,2,1), nn.GroupNorm(4,16), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(16,8,4,2,1), nn.GroupNorm(2,8), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(8,4,4,2,1), nn.GroupNorm(2,4), nn.ReLU(inplace=True))
        self.conv_in = nn.Sequential(
            nn.Conv2d(7,channels,3,padding=1), nn.GroupNorm(8,channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels,channels,3,padding=1), nn.GroupNorm(8,channels), nn.ReLU(inplace=True))
        self.conv_mid = nn.Sequential(
            nn.Conv2d(channels,channels,3,padding=1), nn.GroupNorm(8,channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels,channels,3,padding=1), nn.GroupNorm(8,channels), nn.ReLU(inplace=True))
        self.conv_mid2 = nn.Sequential(
            nn.Conv2d(channels,channels,3,padding=1), nn.GroupNorm(8,channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels,channels,3,padding=1), nn.GroupNorm(8,channels), nn.ReLU(inplace=True))
        self.conv_out = nn.Conv2d(channels,3,3,padding=1)

    def forward(self, frame, wm_bits):
        wm_feat = self.fc_wm(wm_bits).view(frame.size(0),16,8,8)
        x = torch.cat([frame,self.wm_upsample(wm_feat)],dim=1)
        x = self.conv_in(x); x = self.conv_mid(x) + x; x = self.conv_mid2(x) + x
        residual = torch.tanh(self.conv_out(x)) * 0.18
        return torch.clamp(frame + residual,0,1), residual

class WatermarkDecoder(nn.Module):
    def __init__(self, wm_length=WATERMARK_LENGTH, channels=80):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,channels,3,2,1), nn.GroupNorm(8,channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels,channels,3,2,1), nn.GroupNorm(8,channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels,channels,3,2,1), nn.GroupNorm(8,channels), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4,4)))
        self.fc = nn.Sequential(
            nn.Linear(channels*4*4,256), nn.ReLU(inplace=True),
            nn.Linear(256,wm_length))

    def forward(self, frame):
        return self.fc(self.features(frame).reshape(frame.size(0),-1))

encoder, decoder = WatermarkEncoder().to(device), WatermarkDecoder().to(device)
for path in (ENCODER_PATH, DECODER_PATH):
    if not path.is_file(): raise FileNotFoundError(f'Checkpoint watermark tidak ditemukan: {path}')
encoder.load_state_dict(torch.load(ENCODER_PATH,map_location=device))
decoder.load_state_dict(torch.load(DECODER_PATH,map_location=device))
encoder.eval(); decoder.eval()
print('Watermark checkpoints loaded.')

## 3. Neural Codec — inference only

Tidak ada training pada flow evaluasi. Customer source melatih NeuralCodec, tetapi state dict codec perlu disimpan satu kali sebagai `neural_codec.pth`.

In [ ]:
def ste_quantize(z, levels=16):
    hard = torch.round(z * levels) / levels
    return z + (hard - z).detach()

class NeuralCodec(nn.Module):
    def __init__(self, bottleneck=8):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(3,32,4,2,1), nn.ReLU(inplace=True),
            nn.Conv2d(32,64,4,2,1), nn.ReLU(inplace=True),
            nn.Conv2d(64,bottleneck,3,padding=1))
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(bottleneck,64,4,2,1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64,32,4,2,1), nn.ReLU(inplace=True),
            nn.Conv2d(32,3,3,padding=1))

    def forward(self,x,quant_levels=CODEC_QUANT_LEVELS):
        z = ste_quantize(torch.tanh(self.enc(x)),quant_levels)
        return torch.sigmoid(self.dec(z)),z

neural_codec = NeuralCodec().to(device)
if not NEURAL_CODEC_PATH.is_file():
    raise FileNotFoundError(
        f'Checkpoint Neural Codec tidak ditemukan: {NEURAL_CODEC_PATH}. '
        'Simpan state_dict NeuralCodec dari pipeline customer terlebih dahulu.')
neural_codec.load_state_dict(torch.load(NEURAL_CODEC_PATH,map_location=device))
neural_codec.eval()
print('Neural Codec checkpoint loaded.')

In [ ]:
def extract_frames(video_path,max_frames=MAX_FRAMES):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened(): raise RuntimeError(f'Gagal membuka video: {video_path}')
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    size = (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    frames = []
    while len(frames) < max_frames:
        ok,frame = cap.read()
        if not ok: break
        frame = cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame,FRAME_SIZE,interpolation=cv2.INTER_AREA)
        frames.append(frame.astype(np.float32)/255.0)
    cap.release()
    if not frames: raise RuntimeError('Video tidak memiliki frame.')
    return frames,fps,size

def write_video(frames,output_path,fps):
    output_path = Path(output_path); output_path.parent.mkdir(parents=True,exist_ok=True)
    with tempfile.TemporaryDirectory() as tmp:
        for i,frame in enumerate(frames):
            u8 = np.clip(frame*255,0,255).astype(np.uint8)
            cv2.imwrite(str(Path(tmp)/f'frame_{i:05d}.png'),cv2.cvtColor(u8,cv2.COLOR_RGB2BGR))
        subprocess.run([
            'ffmpeg','-y','-loglevel','error','-framerate',str(fps),
            '-i',str(Path(tmp)/'frame_%05d.png'),'-an','-c:v','ffv1',str(output_path)
        ],check=True)
    if not output_path.is_file() or output_path.stat().st_size == 0:
        raise RuntimeError(f'Output video tidak valid: {output_path}')

## 4. Single-pass flow + retry extension point

Flow utama tidak memiliki retry. Jika nanti aturan retry sudah jelas, wrapper dapat memanggil `process_video_once()` tanpa mengubah pipeline inti.

In [ ]:
def process_video_once(input_video):
    input_video = Path(input_video)
    frames,fps,original_size = extract_frames(input_video)
    frames_t = torch.tensor(np.stack(frames),dtype=torch.float32,device=device).permute(0,3,1,2)
    wm_batch = GROUND_TRUTH_WM.repeat(frames_t.size(0),1)

    # INSERT
    with torch.no_grad():
        watermarked_t,_ = encoder(frames_t,wm_batch)
    watermarked = watermarked_t.permute(0,2,3,1).cpu().numpy()
    watermarked_path = OUTPUT_DIR/f'{input_video.stem}_watermarked.mkv'
    write_video(watermarked,watermarked_path,fps)

    # NEURAL CODEC
    recon_chunks=[]; latent_chunks=[]
    with torch.no_grad():
        for start in range(0,frames_t.size(0),8):
            recon,latent = neural_codec(watermarked_t[start:start+8])
            recon_chunks.append(recon.cpu().numpy()); latent_chunks.append(latent.cpu().numpy())
    recon = np.concatenate(recon_chunks).transpose(0,2,3,1)
    latent = np.concatenate(latent_chunks)
    reconstructed_path = OUTPUT_DIR/f'{input_video.stem}_neural_reconstructed.mkv'
    bitstream_path = OUTPUT_DIR/f'{input_video.stem}_neural_latent.npz'
    np.savez_compressed(
        bitstream_path,
        latent=np.round(latent*CODEC_QUANT_LEVELS).astype(np.int16),
        quant_levels=np.array(CODEC_QUANT_LEVELS), shape=np.array(latent.shape))
    write_video(recon,reconstructed_path,fps)

    # READ BACK
    decoded_frames,_,_ = extract_frames(reconstructed_path,max_frames=len(recon))
    decoded_t = torch.tensor(np.stack(decoded_frames),dtype=torch.float32,device=device).permute(0,3,1,2)
    with torch.no_grad():
        probs = torch.sigmoid(decoder(decoded_t)).mean(dim=0).cpu().numpy()
    raw_bits = (probs >= 0.5).astype(np.int64)
    payload = ecc_majority_vote(raw_bits)
    decoded_text = bits_to_text(payload)
    raw_ber = float(np.mean(raw_bits != GROUND_TRUTH_BITS))
    detected = decoded_text == WATERMARK_TEXT

    return {
        'input':str(input_video),
        'watermarked_video':str(watermarked_path),
        'neural_bitstream':str(bitstream_path),
        'neural_reconstructed_video':str(reconstructed_path),
        'payload_text':WATERMARK_TEXT,
        'payload_binary':''.join(map(str,PAYLOAD)),
        'encoded_bits':WATERMARK_LENGTH,
        'raw_ber':raw_ber,
        'decoded_text':decoded_text,
        'detected':bool(detected),
        'status':'SUCCESS' if detected else 'FAILED',
        'frames':len(frames),
        'fps':fps,
        'original_size':original_size,
    }

def process_video_with_retry(*args,**kwargs):
    raise NotImplementedError('Retry belum diaktifkan pada scope ini.')

In [ ]:
result = process_video_once(INPUT_VIDEO)

print('\n=== FINAL RESULT ===')
for key,value in result.items():
    print(f'{key}: {value}')

print(
    '\nTRUE — watermark sabila berhasil dibaca kembali setelah Neural Codec.'
    if result['detected']
    else '\nFALSE — watermark sabila tidak berhasil dibaca kembali.'
)

## Bukti output

- `decoded_text == "sabila"` adalah kriteria utama SUCCESS.
- Binary payload asli = 48 bit: `01110011 01100001 01100010 01101001 01101100 01100001`.
- Encoded ECC = 144 bit pada baseline customer.
- `raw_ber` adalah metrik tambahan sebelum ECC.
- Retry belum aktif.